# 🚀 GHOST ARCHITECT: GEMMA-3 TRAINING ON KAGGLE RTX PRO 6000

**Mode**: Offline (no internet after CELL 1)  
**GPU**: RTX Pro 6000 (24GB VRAM)  
**Strategy**: 2-Phase (Text Foundation → Vision Fine-tune)  
**Stack**: QLoRA + DoRA + rsLoRA (Trinity)  
**Data**: 5,287 examples (5000 synthetic text + 287 real UI screenshots)  
**Expected Time**: ~70-80 minutes total


## CELL 1: Install Dependencies (Offline Wheels)

In [ ]:
# Install from pre-downloaded wheels (no internet needed after this)
!pip install --no-index \
  --find-links=/kaggle/input/datasets/harshilmaks/unsloth-offline-wheels/wheels \
  unsloth trl peft accelerate bitsandbytes transformers datasets pillow numpy xformers

print("✅ Dependencies installed from offline wheels")

## CELL 2: Unzip Pre-downloaded Gemma Model

In [ ]:
# Unzip the pre-downloaded Gemma-3 model
!unzip -q /kaggle/input/datasets/harshilmaks/gemma-3-model-offline/gemma3_model.zip -d /kaggle/working/ 2>/dev/null || echo "⚠️  Zip may already exist"
!ls -lh /kaggle/working/gemma3_model/ | head -10

print("✅ Gemma-3 model extracted")

## CELL 3: Environment Setup & PyTorch Config

In [ ]:
import os
import torch
from pathlib import Path

# Critical environment variables for GPU memory safety
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"

# Force safe attention backend (avoid flex-attention bugs on this stack)
os.environ["XFORMERS_DISABLED"] = "1"
os.environ["DISABLE_FLEX_ATTENTION"] = "1"

# PyTorch optimizations for RTX Pro 6000
torch.backends.cuda.matmul.allow_tf32 = True
if hasattr(torch.backends, "cuda") and hasattr(torch.backends.cuda, "enable_math_sdp"):
    torch.backends.cuda.enable_math_sdp(True)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    print("✅ Attention backend: math SDPA (safe)")

# Verify GPU
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"✅ CUDA: {torch.version.cuda}")

## CELL 4: Load & Prepare Dataset (Text + Vision Split)

In [ ]:
import json
from pathlib import Path
from datasets import Dataset, Image
from PIL import Image as PILImage

DATA_ROOT = Path("/kaggle/input/datasets/harshilmaks/ghost-architect-data/data")
IMG_DIR = DATA_ROOT / "ui_screenshots"
JSON_PATH = DATA_ROOT / "dataset_merged.json"

print(f"📂 Loading dataset from: {DATA_ROOT}")

with open(JSON_PATH, "r") as f:
    raw_data = json.load(f)

print(f"📊 Total examples: {len(raw_data)}")

# Split into text-only and multimodal
text_only_rows = []
vision_rows = []

for item in raw_data:
    instruction = str(item.get("instruction", "")).strip()
    output = str(item.get("output", "")).strip()
    formatted_text = f"### Instruction:\n{instruction}\n\n### Output:\n{output}"

    # Check if image exists
    img_name = Path(str(item.get("image_path", ""))).name
    img_path = IMG_DIR / img_name if img_name else None

    if img_path and img_path.exists():
        # ✅ FIX: Load PIL image now, not later
        try:
            pil_image = PILImage.open(str(img_path)).convert("RGB")
            vision_rows.append({
                "text": formatted_text,
                "image": pil_image,
            })
        except Exception as e:
            print(f"⚠️  Skipped corrupted image: {img_name}")
    else:
        text_only_rows.append({
            "text": formatted_text,
        })

# Create HF Datasets
text_dataset = Dataset.from_list(text_only_rows)
vision_dataset = Dataset.from_list(vision_rows)

print(f"\n✅ Text-only samples (synthetic): {len(text_dataset)}")
print(f"✅ Vision samples (real UI): {len(vision_dataset)}")
print(f"\n📝 Sample text example:")
print(text_dataset[0]["text"][:200])
print(f"\n🖼️  Sample vision example:")
print(f"   Text: {vision_dataset[0]['text'][:100]}...")
print(f"   Image: {type(vision_dataset[0]['image'])} size={vision_dataset[0]['image'].size}")

## CELL 5: Load Gemma-3 + Trinity Stack (QLoRA + DoRA + rsLoRA)

In [ ]:
from unsloth import FastVisionModel, is_bfloat16_supported
from unsloth.trainer import UnslothVisionDataCollator

print("🚀 Loading Gemma-3-12B-IT with Trinity Stack...")
print(f"   Model path: /kaggle/working/gemma3_model")
print(f"   Load in 4-bit: True (QLoRA compression)")
print(f"   Attention: eager (safe, math-based)")

model, processor = FastVisionModel.from_pretrained(
    model_name="/kaggle/working/gemma3_model",  # Local path, no internet
    load_in_4bit=True,  # QLoRA: compresses 48GB → 12GB
    attn_implementation="eager",  # ✅ FIX: Explicit safe attention backend
)

print("\n✅ Base model loaded")

# Apply Trinity Stack (exactly like Modal setup)
model = FastVisionModel.get_peft_model(
    model,
    # Vision: Phase 1 disabled, Phase 2 will enable
    finetune_vision_layers=False,  # PHASE 1: text-only learning
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    # LoRA configuration
    r=64,  # High rank (enabled by rsLoRA stability)
    lora_alpha=32,
    lora_dropout=0,  # 0 = Unsloth fast-patches ALL layers
    bias="none",
    random_state=42,
    # Trinity Stack
    use_rslora=True,   # ✅ Rank stabilization (prevents gradient collapse at r=64)
    use_dora=True,     # ✅ Weight decomposition (magnitude + direction)
)

print("\n✅ Trinity Stack Applied:")
print(f"   ✓ QLoRA (4-bit quantization): 12.4B → ~7.6GB")
print(f"   ✓ DoRA (weight decomposition): Enabled")
print(f"   ✓ rsLoRA (rank stabilization): Enabled (r=64)")
print(f"   ✓ Vision fine-tuning: Disabled for Phase 1")
print(f"\n📊 Trainable parameters:")
model.print_trainable_parameters()

## CELL 6: Text Formatter Function

In [ ]:
def format_text_prompt(examples):
    """
    Format text examples for training.
    Unsloth requires a list of formatted texts.
    """
    return [example["text"] for example in examples]

# Test
test_sample = format_text_prompt([text_dataset[0]])
print("✅ Text formatter test:")
print(test_sample[0][:300])

## CELL 7: PHASE 1 — Text-Only Training (5000 samples, ~40 min)

In [ ]:
from trl import SFTTrainer, SFTConfig

print("\n" + "="*70)
print("🔥 PHASE 1: TEXT-ONLY FOUNDATION TRAINING")
print("="*70)
print(f"Samples: {len(text_dataset)} (synthetic descriptions)")
print(f"Objective: Learn SQL schema generation from text")
print(f"Expected time: ~40 minutes\n")

trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    data_collator=UnslothVisionDataCollator(model, processor),
    train_dataset=text_dataset,
    args=SFTConfig(
        output_dir="./phase1_checkpoint",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,  # Effective batch = 8
        num_train_epochs=1,  # One pass over 5000 samples
        learning_rate=2e-4,
        lr_scheduler_type="cosine",  # Cosine decay (like Modal)
        warmup_ratio=0.1,
        optim="adamw_8bit",  # Memory-efficient optimizer
        save_strategy="epoch",
        logging_steps=50,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),  # RTX Pro 6000 supports bf16
        gradient_checkpointing=True,
        max_grad_norm=1.0,
        seed=42,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=2048,  # Gemma-3 runtime cap
    ),
)

print("🚀 Starting Phase 1 training...\n")
trainer.train()
print("\n✅ Phase 1 complete!")

## CELL 8: Reload Model + Enable Vision Fine-Tuning

In [ ]:
print("\n" + "="*70)
print("🎯 TRANSITIONING TO PHASE 2: VISION FINE-TUNING")
print("="*70)

# Reload model with vision layers now trainable
model = FastVisionModel.get_peft_model(
    model,
    # Vision: NOW ENABLE
    finetune_vision_layers=True,  # ✅ PHASE 2: Fine-tune vision encoder (SigLIP)
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    # LoRA config (same as Phase 1)
    r=64,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=42,
    use_rslora=True,
    use_dora=True,
)

print("\n✅ Vision layers now trainable")
print(f"📊 Updated trainable parameters:")
model.print_trainable_parameters()

## CELL 9: Vision Formatter Function

In [ ]:
def format_vision_prompt(examples):
    """
    Format vision examples (text + image) for training.
    Unsloth requires a list of formatted texts.
    Images are handled separately via the data collator.
    """
    return [example["text"] for example in examples]

# Test
test_vision = format_vision_prompt([vision_dataset[0]])
print("✅ Vision formatter test:")
print(f"Text: {test_vision[0][:200]}...")
print(f"Image attached: {vision_dataset[0]['image'].size}")

## CELL 10: PHASE 2 — Vision Fine-Tuning (287 samples, ~30 min)

In [ ]:
print("\n" + "="*70)
print("🔥 PHASE 2: VISION FINE-TUNING")
print("="*70)
print(f"Samples: {len(vision_dataset)} (real UI screenshots)")
print(f"Objective: Ground schema generation in actual UI images")
print(f"Learning rate: 5e-5 (lower for fine-tuning)")
print(f"Expected time: ~30 minutes\n")

trainer = SFTTrainer(
    model=model,
    processing_class=processor,
    data_collator=UnslothVisionDataCollator(model, processor),  # ✅ Handles vision data properly
    train_dataset=vision_dataset,
    args=SFTConfig(
        output_dir="./phase2_checkpoint",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,  # Smaller batch for fine-tuning
        num_train_epochs=2,  # Two passes over 287 samples
        learning_rate=5e-5,  # ✅ Lower LR for vision fine-tuning (not catastrophic forgetting)
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        optim="adamw_8bit",
        save_strategy="epoch",
        logging_steps=10,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        gradient_checkpointing=True,
        max_grad_norm=1.0,
        seed=42,
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=2048,
    ),
)

print("🚀 Starting Phase 2 training...\n")
trainer.train()
print("\n✅ Phase 2 complete!")

## CELL 11: Save Final Adapter

In [ ]:
print("\n" + "="*70)
print("💾 SAVING FINAL ADAPTER")
print("="*70)

FINAL_ADAPTER_DIR = "/kaggle/working/final_adapter"

model.save_pretrained(FINAL_ADAPTER_DIR)
processor.save_pretrained(FINAL_ADAPTER_DIR)

print(f"\n✅ Adapter saved to: {FINAL_ADAPTER_DIR}")

# List files
import os
adapter_files = os.listdir(FINAL_ADAPTER_DIR)
total_size = sum(os.path.getsize(f"{FINAL_ADAPTER_DIR}/{f}") for f in adapter_files) / 1024 / 1024

print(f"\n📂 Files ({len(adapter_files)} items, {total_size:.1f} MB):")
for f in sorted(adapter_files):
    size = os.path.getsize(f"{FINAL_ADAPTER_DIR}/{f}") / 1024 / 1024
    print(f"   - {f} ({size:.1f} MB)")

## CELL 12: Verify Training Completion & Summary

In [ ]:
print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)

print(f"""
📊 SUMMARY:
   Phase 1: 5,000 text examples (1 epoch) ✓
   Phase 2: 287 vision examples (2 epochs) ✓
   
🏆 ACHIEVEMENTS:
   ✓ QLoRA: 12.4B model on 24GB VRAM
   ✓ DoRA: Weight decomposition for precision
   ✓ rsLoRA: Rank-64 stability
   ✓ Vision: SigLIP encoder fine-tuned
   ✓ Language: Full Gemma-3 attention layers trained
   
📦 OUTPUT:
   Adapter: {FINAL_ADAPTER_DIR}
   Size: {total_size:.1f} MB
   
🚀 NEXT STEPS:
   1. Download final_adapter folder
   2. Move to: output/adapters/trinity_kaggle/
   3. Run: make export --adapter-dir output/adapters/trinity_kaggle/
   4. Test with: streamlit run src/app.py
""")

print("\n✅ Ready to export to GGUF for Ollama deployment!")

## CELL 13 (OPTIONAL): Quick Inference Test

In [ ]:
# Optional: Quick inference test to verify model works
from unsloth import FastVisionModel

print("🧪 Quick inference test...\n")

# Prepare for inference
FastVisionModel.for_inference(model)

# Use a sample from text dataset
sample_text = text_dataset[0]["text"]
sample_instruction = "Generate PostgreSQL schema from this UI"

inputs = processor(
    text=f"### Instruction:\n{sample_instruction}\n\n### Output:",
    images=None,  # Text-only for this test
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
    )

generated = processor.decode(outputs[0], skip_special_tokens=True)
print("📝 Generated Schema Sample:")
print(generated[-300:])
print("\n✅ Inference working!")